# YOLO11 Pose Estimation

This notebook detects human poses, extracts **17 anatomical keypoints** (eyes, ears, shoulders, elbows, wrists, hips, knees, ankles), and renders the body skeleton using **Ultralytics YOLO11 Pose**.

In [2]:
from pathlib import Path
from urllib.request import urlretrieve
from ultralytics import YOLO

# Base paths
HOME_DIR = Path.home()
MODEL_DIR = HOME_DIR / ".files" / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Download YOLO11n-pose model if not already present
MODEL_PATH = MODEL_DIR / "yolo11n-pose.pt"
MODEL_URL = "https://github.com/ultralytics/assets/releases/download/v8.3.0/yolo11n-pose.pt"

if not MODEL_PATH.exists():
    print("Downloading YOLO11n-pose model weights...")
    urlretrieve(MODEL_URL, MODEL_PATH)
    print("Download completed!")
else:
    print("YOLO11n-pose model already cached at:", MODEL_PATH)

# Load the pose model
model = YOLO(str(MODEL_PATH))
print("YOLO11n-pose model loaded successfully!")


YOLO11n-pose model already cached at: /Users/startup/.files/models/yolo11n-pose.pt
YOLO11n-pose model loaded successfully!


In [3]:
import cv2
import matplotlib.pyplot as plt


In [4]:
# Project directories setup (works whether run from root or inside notebooks/)
PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATASET_DIR = PROJECT_DIR / "dataset"
IMAGE_DIR = DATASET_DIR / "images"
OUTPUT_DIR = PROJECT_DIR / "runs" / "pose_estimation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

image_path = IMAGE_DIR / "construction_worker.jpg"

if not image_path.exists():
    raise FileNotFoundError(f"Image not found: {image_path}")

print("Input Image:", image_path)
print("Output Directory:", OUTPUT_DIR)


Input Image: /Users/startup/projects/yolotest/dataset/images/construction_worker.jpg
Output Directory: /Users/startup/projects/yolotest/runs/pose_estimation


In [5]:
# COCO Standard 17 Human Anatomical Keypoints
KEYPOINT_NAMES = [
    "Nose",          # 0
    "Left Eye",      # 1
    "Right Eye",     # 2
    "Left Ear",      # 3
    "Right Ear",     # 4
    "Left Shoulder", # 5
    "Right Shoulder",# 6
    "Left Elbow",    # 7
    "Right Elbow",   # 8
    "Left Wrist",    # 9
    "Right Wrist",   # 10
    "Left Hip",      # 11
    "Right Hip",     # 12
    "Left Knee",     # 13
    "Right Knee",    # 14
    "Left Ankle",    # 15
    "Right Ankle"     # 16
]

print(f"Loaded {len(KEYPOINT_NAMES)} COCO keypoint definitions.")


Loaded 17 COCO keypoint definitions.


In [6]:
# Run YOLO pose inference
results = model.predict(
    source=str(image_path),
    conf=0.25,
    imgsz=640,
    device="cpu"
)

print("Pose inference completed!")



image 1/1 /Users/startup/projects/yolotest/dataset/images/construction_worker.jpg: 640x608 1 person, 199.6ms
Speed: 7.9ms preprocess, 199.6ms inference, 1.1ms postprocess per image at shape (1, 3, 640, 608)
Pose inference completed!


In [7]:
# Extract and inspect detected Keypoints
for person_idx, (box, kpts) in enumerate(zip(results[0].boxes, results[0].keypoints)):
    box_conf = box.conf[0].item()
    print(f"=== Person {person_idx + 1} (Box Confidence: {box_conf:.4f}) ===")
    
    # kpts.xy is a PyTorch Tensor of shape (1, 17, 2)
    xy_coords = kpts.xy[0].tolist()
    
    # kpts.conf is a PyTorch Tensor of shape (1, 17)
    conf_scores = kpts.conf[0].tolist() if kpts.conf is not None else [1.0] * 17
    
    for kpt_idx, ((x, y), score) in enumerate(zip(xy_coords, conf_scores)):
        name = KEYPOINT_NAMES[kpt_idx]
        print(f"  {kpt_idx:2d}. {name:15s}: X={x:6.1f}, Y={y:6.1f} | Conf: {score:.4f}")
    print("-" * 55)


=== Person 1 (Box Confidence: 0.9161) ===
   0. Nose           : X= 273.8, Y= 161.0 | Conf: 0.9985
   1. Left Eye       : X= 292.8, Y= 137.7 | Conf: 0.9953
   2. Right Eye      : X= 244.4, Y= 137.0 | Conf: 0.9985
   3. Left Ear       : X= 312.4, Y= 157.4 | Conf: 0.7078
   4. Right Ear      : X= 195.8, Y= 159.3 | Conf: 0.9762
   5. Left Shoulder  : X= 369.5, Y= 307.7 | Conf: 0.9812
   6. Right Shoulder : X= 114.9, Y= 306.7 | Conf: 0.9865
   7. Left Elbow     : X= 415.1, Y= 489.2 | Conf: 0.1480
   8. Right Elbow    : X=  78.8, Y= 492.1 | Conf: 0.1462
   9. Left Wrist     : X= 371.1, Y= 452.8 | Conf: 0.1296
  10. Right Wrist    : X= 153.4, Y= 478.6 | Conf: 0.1262
  11. Left Hip       : X= 312.1, Y= 495.0 | Conf: 0.0043
  12. Right Hip      : X= 157.9, Y= 495.0 | Conf: 0.0040
  13. Left Knee      : X= 348.6, Y= 432.7 | Conf: 0.0004
  14. Right Knee     : X= 214.3, Y= 437.3 | Conf: 0.0004
  15. Left Ankle     : X= 268.0, Y= 434.0 | Conf: 0.0002
  16. Right Ankle    : X= 262.1, Y= 448.2 | Co

In [10]:
type(kpts.xy[0])

torch.Tensor

In [8]:
type(results[0].boxes)

ultralytics.engine.results.Boxes

In [ ]:
# Render and visualize skeleton overlay
result_image = results[0].plot()

plt.figure(figsize=(12, 8))
plt.imshow(cv2.cvtColor(result_image, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title("YOLO11n Pose Estimation (Keypoints & Skeleton Overlay)")
plt.show()


In [ ]:
# Save the annotated pose image to runs directory
output_file = OUTPUT_DIR / image_path.name
cv2.imwrite(str(output_file), result_image)
print("Pose detection output saved to:", output_file)
